# Lab1: Fine-tuning Large Transformer Models with Amazon SageMaker

### Distributed Sequence Classification with `Trainer` and the `clinc_oos` dataset

Welcome to our end-to-end multi-class Text-Classification example. In this demo, we will use the Hugging Faces `transformers` and `datasets` library together with a Amazon Python SDK  to fine-tune a pre-trained transformer for multi-class text classification using distributed training. In particular, the pre-trained model will be fine-tuned using the `clinc_oos` dataset. To get started, we need to set up the environment with a few prerequisite steps, for permissions, configurations, and so on. 

If you are new to Amazon SageMaker you can check out workshop 1: [getting started with Amazon SageMaker](../workshop_1_getting_started_with_amazon_sagemaker/).


# Introduction

_**NOTE: You can run this demo in Sagemaker Studio, your local machine or Sagemaker Notebook Instances**_

# Development Environment and Permissions 

## Installation

_*Note:* we only install the required libraries from Hugging Face and AWS. You will also need PyTorch or Tensorflow if you haven´t installed one of these frameworks._

In [ ]:
%%capture
%pip install "sagemaker>=2.80.0" huggingface_hub --upgrade -q


In [ ]:
import sagemaker.huggingface

## Permissions

_If you are going to use Sagemaker in a local environment, you need access to an IAM Role with the required permissions for Sagemaker. You can find more about it [here](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-roles.html)._

In [ ]:
import sagemaker
import boto3
sess = sagemaker.Session()
# sagemaker session bucket -> used for uploading data, models and logs
# sagemaker will automatically create this bucket if it not exists
sagemaker_session_bucket=None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

## Creating an Estimator and starting a training job

List of supported models: https://huggingface.co/models?library=pytorch,transformers&sort=downloads

### Pushing our model to the Hugging Face Hub

To push our model to the [Hugging Face Hub](https://huggingface.co/models), we'll to use the `push_to_hub()` method of the `Trainer` in the `transformers` library ([docs](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.push_to_hub)). The `train.py` script that accompanies this notebook provides the relevant parameters to `TrainingArguments`, including a [Hugging Face access token](https://huggingface.co/settings/tokens) associated with your account, the repository name for the final model, and the saving strategy to indicate how often to push a checkpoint to the Hub.

You can find documentation for these parameters [here](https://huggingface.co/docs/transformers/main_classes/trainer).

We are going to provide our HF token securely with out exposing it to the public using the `notebook_login()` function from the `huggingface_hu`b library. But be careful your token will still be visible inside the logs of the training job! If you run `huggingface_estimator.fit(...,wait=True)` you will see the token in the logs. A better way of providing your HF_TOKEN to your training jobs would be using AWS Secret Manager:

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

We can now define our hyperparemeters and will use [roberta-large](https://huggingface.co/roberta-large) as pre-trained model and the `clins_oos` dataset:

In [ ]:
from sagemaker.huggingface import HuggingFace
from huggingface_hub import HfFolder

# hyperparameters, which are passed into the training job
hyperparameters={'epochs': 5,                                                # number of training epochs
                 'train_batch_size': 16,                                     # batch size for training
                 'eval_batch_size': 16,                                      # batch size for evaluation
                 'learning_rate': 2e-5,                                      # learning rate used during training
                 'model_id':'roberta-large',                                 # pre-trained model
                 'dataset_id':'clinc_oos',                                   # dataset id 
                 'dataset_config':'plus',                                    # dataset configuration
                 'fp16': True,                                               # Whether to use 16-bit (mixed) precision training
                 'push_to_hub': True,                                        # Defines if we want to push the model to the hub
                 'hub_model_id': 'roberta-large-finetuned-clinc',            # The model id of the model to push to the hub
                 'hub_strategy': 'every_save',                               # The strategy to use when pushing the model to the hub
                 'hub_token': HfFolder.get_token()                           # HuggingFace token to have permission to push
                }

# configuration for running training on smdistributed Data Parallel
distribution = {'smdistributed':{'dataparallel':{ 'enabled': True }}}

In [ ]:
# define Training Job Name 
job_name = f'huggingface-workshop'

# create the Estimator
huggingface_estimator = HuggingFace(
    entry_point          = 'train.py',        # fine-tuning script used in training jon
    source_dir           = './scripts',       # directory where fine-tuning script is stored
    instance_type        = 'ml.p3.16xlarge',  # instances type used for the training job
    instance_count       = 1,                 # the number of instances used for training
    base_job_name        = job_name,          # the name of the training job
    volume_size          = 300,               # increase size for storing artifacts
    role                 = role,              # Iam role used in training job to access AWS ressources, e.g. S3
    transformers_version = '4.17',            # the transformers version used in the training job
    pytorch_version      = '1.10',            # the pytorch_version version used in the training job
    py_version           = 'py38',            # the python version used in the training job
    hyperparameters      = hyperparameters,   # the hyperparameter used for running the training job
    distribution         = distribution,      # set up distributed training data parallelism
)

In [ ]:
# starting the train job with our uploaded datasets as input
huggingface_estimator.fit(wait=False)

Since we are using the Hugging Face Hub integration with Tensorboard, we can inspect our progress directly on the Hub, as well as testing checkpoints during the training.

You can find the URL for the model on the Hub by running the following cell:

In [ ]:
# skip or comment this out if you're not pushing your model to the Hub
from huggingface_hub import HfApi

whoami = HfApi().whoami()
username = whoami['name']

print(f"https://huggingface.co/{username}/{hyperparameters['hub_model_id']}")

## Deploying the endpoint

To deploy our endpoint, we call `deploy()` on our HuggingFace estimator object, passing in our desired number of instances and instance type.

In [ ]:
predictor = huggingface_estimator.deploy(1,"ml.c5.xlarge")

Then, we use the returned predictor object to call the endpoint. We will send a few hundred requests with a sequence length of 128 to get a estimation of the latency.

In [ ]:
sentiment_input= {"inputs": "Harry believes it, although no one else believes that Sally is innocent." * 9} # generates 128 seq length input

for i in range(200):
    predictor.predict(sentiment_input)

We can now take a look at cloudwatch to get our monitoring metrics. 

In [ ]:
print(f"https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#metricsV2:graph=~(metrics~(~(~'AWS*2fSageMaker~'ModelLatency~'EndpointName~'{predictor.endpoint_name}~'VariantName~'AllTraffic))~view~'timeSeries~stacked~false~region~'us-east-1~start~'-PT10M~end~'P0D~stat~'p99~period~300);query=~'*7bAWS*2fSageMaker*2cEndpointName*2cVariantName*7d*20{predictor.endpoint_name}")

Finally, we delete the inference endpoint.

In [ ]:
predictor.delete_model
predictor.delete_endpoint()

# Create performance chart

Here we gather the results our model achieved on the validation set, along with some model metrics like the 99th latency percentile and the size of the model on disk:

In [ ]:
from visualize import plot_metrics
%matplotlib inline

metrics = {"roberta-large": {"time_p99_ms": 322, "accuracy": 0.9729,"size_mb":1322}}

plot_metrics(metrics, "roberta-large")